# Problem 3 — Phase 3: Slack `C` pilot (random edges)

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 2](hierarchical_problem3_phase2_truncated_svd.ipynb) · **Next:** [Phase 4](hierarchical_problem3_phase4_kernel_pilot.ipynb)

**Goal:** Get a **directional** sense of the linear SVM slack parameter **`C`** without fitting all ~100 binary edges for every value. We **randomly sample** `PILOT_N_EDGES` edges (trainable on the train split), fit **only those** edges for each `C` in `C_GRID`, and report **pooled / macro F1** on the **test** split over that subset.

**Full-tree** tuning (every edge × `C`) lives in [`archive/hierarchical_problem3_phase3_by_depth_full.ipynb`](archive/hierarchical_problem3_phase3_by_depth_full.ipynb).

In [9]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_classifier import MultilabelHierarchyRouter, svm_linear_binary_edge_factory
from hierarchical_summary_metrics import fit_binary_edges_subset, pooled_edge_f1_stats
from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import BinaryEdgeSpec, TopicTree, binary_edge_specs, load_topic_tree


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
SPLIT = "test"
RESTRICT = True
MAX_FEATURES = 8000

PILOT_N_EDGES = 20
PILOT_SEED = 42
C_GRID = [1e-2, 0.1, 1.0, 10.0]

print("BASE", BASE)
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

ImportError: cannot import name 'fit_binary_edges_subset' from 'hierarchical_summary_metrics' (/Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/hierarchical_summary_metrics.py)

### Sample pilot edges

Keep specs where the **train** split has at least two classes; then sample **`PILOT_N_EDGES`** with **`PILOT_SEED`**. Writes **`problem3_phase3_pilot_edges.json`**.

In [ ]:
def eligible_specs_for_pilot(
    pool, tree: Path, *, restrict: bool
) -> list[BinaryEdgeSpec]:
    out: list[BinaryEdgeSpec] = []
    for spec in binary_edge_specs(tree):
        _, ytr = pool.binary_edge_dataset(
            spec.parent,
            spec.child,
            "train",
            restrict_to_parent_subtree=restrict,
        )
        if len(set(ytr)) >= 2:
            out.append(spec)
    return out


eligible = eligible_specs_for_pilot(mldata, tree, restrict=RESTRICT)
rng = random.Random(PILOT_SEED)
k = min(PILOT_N_EDGES, len(eligible))
pilot_specs = rng.sample(eligible, k=k) if eligible else []

pilot_payload = [
    {"parent": s.parent, "child": s.child, "depth": s.depth} for s in pilot_specs
]
pilot_path = BASE / "problem3_phase3_pilot_edges.json"
with open(pilot_path, "w") as f:
    json.dump(pilot_payload, f, indent=2)
print(f"Eligible {len(eligible)}; pilot {len(pilot_specs)} edges -> {pilot_path}")
pilot_tuples = [(s.parent, s.child) for s in pilot_specs]

### Pilot loop over `C`

For each **`C`**, a fresh router; **[`fit_binary_edges_subset`](hierarchical_summary_metrics.py)** on the pilot list; **[`pooled_edge_f1_stats`](hierarchical_summary_metrics.py)** on the same edges on **`SPLIT`**. Set **`RUN_PILOT = False`** to skip after the first run.

In [ ]:
RUN_PILOT = True

rows: list[dict] = []
if not pilot_specs:
    print("No pilot edges; check eligibility / data.")
elif not RUN_PILOT:
    print("SKIP (RUN_PILOT=False)")
else:
    for C in C_GRID:
        print("\n" + "=" * 60, flush=True)
        print(f"C = {C}", flush=True)
        fac = svm_linear_binary_edge_factory(
            max_features=MAX_FEATURES,
            text_vectorizer="tfidf",
            svm_kw=dict(C=float(C), dual=False, max_iter=8000),
        )
        router = MultilabelHierarchyRouter(tree, fac)
        t0 = time.perf_counter()
        fit_binary_edges_subset(
            router,
            mldata,
            pilot_specs,
            verbose=True,
            restrict_to_parent_subtree=RESTRICT,
        )
        wall = time.perf_counter() - t0
        st = pooled_edge_f1_stats(
            router,
            mldata,
            pilot_tuples,
            SPLIT,
            restrict_to_parent_subtree=RESTRICT,
        )
        rows.append(
            {
                "C": C,
                "pooled_micro_f1": st["pooled_micro_f1"],
                "macro_f1": st["macro_f1"],
                "pos_weighted_f1": st["pos_weighted_f1"],
                "n_edges_scored": st["n_edges_used"],
                "wall_time_sec": wall,
            }
        )

    df = pd.DataFrame(rows)
    display(df.round(4))
    out_csv = BASE / "problem3_phase3_pilot_c_results.csv"
    df.to_csv(out_csv, index=False)
    print("Wrote", out_csv)
    if not df.empty:
        best = df.loc[df["pooled_micro_f1"].idxmax()]
        print(
            f"Best C by pooled_micro_f1 on pilot: C={best['C']:.4g} "
            f"(pooled_micro_f1={best['pooled_micro_f1']:.4f})"
        )